# 07 — Prediction maps and deployment readiness

* Prediction maps from saved prediction tables (validation from notebook 04/05; test from notebook 06 if run).
  No model is re-run for the maps and nothing is invented: if a table does not exist, the cell says so.
* Inference latency benchmark on your GPU (§3, **GPU**, opt-in).
* Cursor smoothing trade-off and safe handling of lost tracking (§4).
* A live-webcam demo cell that is **disabled** by default (§5).
* How to integrate the model into the astronomy map (§6).

In [ ]:
# --- Workspace setup -----------------------------------------------------------
# WHAT: makes the `src` package importable and loads training_config.yaml.
# WHY : every notebook uses the same code and the same configuration.
# PATHS: resolved from this notebook's folder, never from the terminal's cwd.
import sys
from pathlib import Path

WORKSPACE = Path.cwd().resolve()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent
if not (WORKSPACE / "src" / "config.py").is_file():
    raise RuntimeError(f"Open this notebook from training_workspace/notebooks (cwd = {Path.cwd()})")
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from src.config import load_config
cfg = load_config(WORKSPACE / "training_config.yaml")
cfg.paths.ensure_output_dirs()
W, H = cfg.screen_px
print("Workspace     :", cfg.paths.workspace)
print("Dataset root  :", cfg.paths.dataset_root, "(read-only)")
print("Screen        :", W, "x", H, "px")

In [ ]:
import json, numpy as np, pandas as pd, torch
from IPython.display import Image, display
from src.visualization import plot_prediction_map, plot_vector_field, interactive_prediction_map
SOURCES = {"validation": None, "test": cfg.paths.predictions / "test" / "test_predictions.parquet"}
dec = cfg.paths.reports / "model_selection_decision.json"
if dec.is_file():
    run = json.loads(dec.read_text())["run"]
    SOURCES["validation"] = cfg.paths.experiments / run / "val_predictions_all_frames.parquet"
tables = {k: pd.read_parquet(v) for k, v in SOURCES.items() if v is not None and Path(v).is_file()}
print({k: len(v) for k, v in tables.items()} or "no prediction tables yet - run notebooks 04 (and 06)")

## 1. Maps (aspect ratio preserved, origin top-left, y downward)

In [ ]:
for split, t in tables.items():
    fd = cfg.paths.figures / "maps" / split
    display(Image(filename=str(plot_prediction_map(t, W, H, fd / "all", f"{split}: all predictions"))))
    display(Image(filename=str(plot_vector_field(t, W, H, fd / "vectors"))))
    print(interactive_prediction_map(t, W, H, fd / f"interactive_{split}.html", f"{split} predictions"))

## 2. Load the exported model

In [ ]:
MODEL_FILE = cfg.paths.checkpoints / "selected" / "gaze_model.pt"
print(MODEL_FILE, "exists:", MODEL_FILE.is_file())

## 3. Inference benchmark — GPU, opt-in
Batch-size-1 latency is what the live cursor experiences; larger batches only show offline throughput.
Timings are synchronised (`torch.cuda.synchronize`). End-to-end uses real raw frames from the **validation** set
and includes YuNet + head pose + normalisation, but not camera capture time.
Target for a usable cursor: ≥15 predictions/s end-to-end (≈ ≤66 ms per frame).

In [ ]:
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    from src.infer import GazePredictor, load_exported
    from src.benchmark import benchmark_model, benchmark_end_to_end
    from src.dataset import load_split
    from src.preprocessing import read_bgr
    model, blob = load_exported(MODEL_FILE, "cuda")
    model_bench = benchmark_model(model, "cuda", "fp16")
    pred = GazePredictor(model, blob, cfg.paths.yunet_model, "cuda")
    frames = [read_bgr(cfg.paths.dataset_root / p) for p in load_split(cfg, "validation")["raw_path"].sample(60, random_state=0)]
    e2e = benchmark_end_to_end(pred, frames)
    from src.utils import write_json
    write_json(cfg.paths.reports / "inference_benchmark.json", {"model": model_bench, "end_to_end": e2e})
    print(json.dumps({"model": model_bench, "end_to_end": e2e}, indent=1))

## 4. Smoothing for the cursor, and lost tracking
Suggested live chain: `GazePredictor` → (optional) `PersonalCalibration.apply` → `OneEuroFilter` →
`TrackingGuard(hold_seconds=0.25)` → cursor. When the face is lost (blink, look-away) the guard holds the
cursor for a quarter second and then reports *no gaze* — it never extrapolates. Dwell-selection timers should
pause while there is no gaze.

In [ ]:
from src.smoothing import FILTERS, smooth_predictions, fixation_jitter
if "validation" in tables:
    rows = [{"filter": n, **fixation_jitter(smooth_predictions(tables["validation"], f), W, H)} for n, f in FILTERS.items()]
    display(pd.DataFrame(rows).round(1))

## 5. Live webcam demo — disabled by default (opens the camera)

In [ ]:
RUN_LIVE_DEMO = False
if RUN_LIVE_DEMO:
    import cv2, time
    from src.infer import GazePredictor
    from src.smoothing import OneEuroFilter, TrackingGuard
    predictor = GazePredictor.from_export(MODEL_FILE, cfg.paths.yunet_model, "cuda")
    filt, guard = OneEuroFilter(min_cutoff=1.0, beta=0.01), TrackingGuard(0.25)
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    try:
        t_end = time.time() + 20
        while time.time() < t_end:
            ok, frame = cap.read()
            if not ok:
                continue
            t = time.time(); r = predictor.predict_frame(frame, t)
            xy = filt(t, np.array([r["x_px"], r["y_px"]])) if r["ok"] else None
            shown = guard.update(t, xy)
            print("no gaze" if shown is None else f"cursor {shown.round(0)}", r.get("reason", ""), end="\r")
    finally:
        cap.release()

## 6. Integrating with the astronomy map
1. Load once: `predictor = GazePredictor.from_export("checkpoints/selected/gaze_model.pt", yunet_path, "cuda")`.
2. Per user, run a 9-point calibration (same layout as `calibration.canonical_layout(9)`), collect ~1 s of
   predictions per point, `PersonalCalibration("ridge_affine", alpha).fit(preds, targets)`, save to JSON.
3. Per frame: `predict_frame` → `calibration.apply` → `OneEuroFilter` → `TrackingGuard` → screen pixels.
4. Map screen pixels to the sky map with the map widget's own pixel → (RA, Dec) transform.
5. Make selectable objects at least as large as the measured error (e.g. the radius that gives ≥ 90 % hits in
   `target_selection_success`) and keep them away from the outer 5 % of the screen, where no training targets exist.
6. The monitor resolution must match the training monitor (2560×1600) or predictions must be rescaled in
   normalised units — and accuracy on a different monitor/camera position must be re-measured.